# Search4Cure.AI: executable walkthrough

Run the cells in order. This notebook calls the project functions to query **real arXiv data**, download one version, extract text by physical PDF page, and create traceable text chunks. Running it again makes new source requests and creates new local retrieval/extraction records.

Use the **Search4Cure (.venv)** kernel after the notebook setup in the README. Saved outputs are snapshots of a particular execution, not evidence that the latest source results are unchanged.

Start Docker Desktop before running the database cells. They create a local password file, start PostgreSQL, and check the connection. No `.env` in `data/` is needed.

The document-storage cell imports real download metadata into PostgreSQL and shows the SQL row as a pandas DataFrame.

In [1]:
import json
import logging
import os
from pathlib import Path
import ssl
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "arxiv_search.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "arxiv_search.py").is_file():
    raise RuntimeError("Open the notebook from the Search4Cure project directory.")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Use an existing system trust store only when this macOS Python lacks a default CA file.
if sys.platform == "darwin" and ssl.get_default_verify_paths().cafile is None:
    system_ca = Path("/etc/ssl/cert.pem")
    if system_ca.is_file():
        os.environ["SSL_CERT_FILE"] = str(system_ca)
        print("Using the system CA bundle; HTTPS verification remains enabled.")

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)

import pymupdf
from arxiv_search import search_one
from download_pdf import download_pdf
from extract_text import extract_text
from chunk_text import chunk_text

QUERY = "all:diabetes"
# The working directory is the project root, including when opened from notebooks/.
OUTPUT_DIR = Path("data/arxiv")
print(f"Python {sys.version.split()[0]}; PyMuPDF {pymupdf.VersionBind}")
print(f"Search query: {QUERY}")
from database import HEALTH_SQL, PASSWORD_FILE, create_password_file, check_connection

from database import initialize_schema
from store_document import DOCUMENT_SQL, store_document, read_document


Using the system CA bundle; HTTPS verification remains enabled.
Python 3.13.12; PyMuPDF 1.28.2
Search query: all:diabetes


## 1. Query publication metadata

This cell contacts the arXiv API for at most one result. Keep first-submission time, version-submission time, and our retrieval time distinct. Unknown fields remain unknown.

In [2]:
print("Querying arXiv...")
search_result = search_one(QUERY)
if search_result["document"] is None:
    raise RuntimeError("No publication matched this query.")
print(json.dumps(search_result, indent=2, ensure_ascii=False))

Querying arXiv...


{
  "query": "all:diabetes",
  "request_url": "https://export.arxiv.org/api/query?search_query=all%3Adiabetes&start=0&max_results=1&sortBy=submittedDate&sortOrder=descending",
  "retrieved_at": "2026-11-04T19:01:35.923600+00:00",
  "document": {
    "arxiv_id": "2611.04405",
    "version": 1,
    "title": "DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for Imbalanced Diabetic Retinopathy Grading via Adaptive SMOTE Fusion",
    "source_url": "http://arxiv.org/abs/2611.04405v1",
    "first_submitted_at": "2026-11-03T06:44:25Z",
    "version_submitted_at": "2026-11-03T06:44:25Z"
  }
}


## 2. Download the exact version

The download requires a known version, streams bytes to disk, and records a SHA-256 fingerprint and a separate PDF retrieval time. The pause respects the API request spacing. Only one importer should run at a time.

In [3]:
print("Waiting 3.1 seconds before the PDF request...")
time.sleep(3.1)
download_result = download_pdf(search_result, OUTPUT_DIR)
print(json.dumps(download_result, indent=2, ensure_ascii=False))

Waiting 3.1 seconds before the PDF request...


PDF saved: data/arxiv/2611.04405v1/932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075.pdf (577377 bytes)


Metadata saved: data/arxiv/2611.04405v1/293574dc028849719a50a818b830f9f0.json


{
  "search": {
    "query": "all:diabetes",
    "request_url": "https://export.arxiv.org/api/query?search_query=all%3Adiabetes&start=0&max_results=1&sortBy=submittedDate&sortOrder=descending",
    "retrieved_at": "2026-11-04T19:01:35.923600+00:00",
    "document": {
      "arxiv_id": "2611.04405",
      "version": 1,
      "title": "DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for Imbalanced Diabetic Retinopathy Grading via Adaptive SMOTE Fusion",
      "source_url": "http://arxiv.org/abs/2611.04405v1",
      "first_submitted_at": "2026-11-03T06:44:25Z",
      "version_submitted_at": "2026-11-03T06:44:25Z"
    }
  },
  "pdf_url": "https://arxiv.org/pdf/2611.04405v1",
  "pdf_retrieved_at": "2026-11-04T19:01:39.267498+00:00",
  "sha256": "932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075",
  "size_bytes": 577377,
  "local_path": "data/arxiv/2611.04405v1/932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075.pdf",
  "status": "downloaded",
  "metada

## 3. Extract text by physical PDF page

The extractor verifies the saved PDF hash before parsing it. It records the parser version/configuration, page numbers starting at 1, empty pages, and any NUL replacements. Each attempt creates its own JSON result; a failed attempt does not replace the source or a previous result.

In [4]:
extraction_result = extract_text(download_result["metadata_path"])
assert extraction_result["download"]["sha256"] == download_result["sha256"]
assert extraction_result["page_count"] == len(extraction_result["pages"])
summary = {
    "arxiv_id": extraction_result["download"]["search"]["document"]["arxiv_id"],
    "version": extraction_result["download"]["search"]["document"]["version"],
    "status": extraction_result["status"],
    "page_count": extraction_result["page_count"],
    "empty_pages": extraction_result["empty_pages"],
    "total_characters": extraction_result["total_characters"],
    "parser": extraction_result["parser"],
    "parser_repaired_pdf": extraction_result["parser_repaired_pdf"],
    "output": extraction_result["metadata_path"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

Verifying PDF hash: data/arxiv/2611.04405v1/932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075.pdf


Page 1/6: 6714 characters


Page 2/6: 7058 characters


Page 3/6: 4269 characters


Page 4/6: 7055 characters


Page 5/6: 6945 characters


Page 6/6: 5573 characters


Extraction record saved: data/arxiv/2611.04405v1/extracted/5182320f27ed4e7c8a76b4ec46606791.json


{
  "arxiv_id": "2611.04405",
  "version": 1,
  "status": "extracted",
  "page_count": 6,
  "empty_pages": [],
  "total_characters": 37614,
  "parser": {
    "name": "PyMuPDF",
    "version": "1.28.2",
    "text_mode": "text",
    "sort": true,
    "ocr": false,
    "page_numbering": "physical PDF pages, starting at 1",
    "nul_handling": "replace NUL with U+FFFD and record the replacement count"
  },
  "parser_repaired_pdf": false,
  "output": "data/arxiv/2611.04405v1/extracted/5182320f27ed4e7c8a76b4ec46606791.json"
}


## 4. Split page text into traceable chunks

A word window keeps up to `MAX_WORDS` non-whitespace runs, with `OVERLAP_WORDS` repeated between adjacent windows. Chunks never cross page boundaries. Character offsets preserve the exact source slice. The chosen word count is a demonstration configuration, not a validated model-token budget or retrieval-quality optimum.

Chunk IDs include document version, PDF hash, parser/chunking configuration, page, and content. Equivalent reruns retain IDs even when retrieval records have new timestamps or filenames.

In [5]:
MAX_WORDS = 100
OVERLAP_WORDS = 20
chunk_result = chunk_text(extraction_result["metadata_path"],
                          max_words=MAX_WORDS, overlap_words=OVERLAP_WORDS)
print(json.dumps({
    "status": chunk_result["status"],
    "chunk_count": chunk_result["chunk_count"],
    "pages_without_text": chunk_result["pages_without_text"],
    "configuration": chunk_result["chunking"],
    "output": chunk_result["metadata_path"],
}, indent=2, ensure_ascii=False))

example = chunk_result["chunks"][0]
source_page = extraction_result["pages"][example["page_number"] - 1]
assert example["text"] == source_page["text"][example["start_char"]:example["end_char"]]
print("First chunk; its exact source-page slice was checked:")
print(json.dumps(example, indent=2, ensure_ascii=False))

Page 1/6: 8 chunks


Page 2/6: 10 chunks


Page 3/6: 6 chunks


Page 4/6: 10 chunks


Page 5/6: 8 chunks


Page 6/6: 8 chunks


Chunk record saved: data/arxiv/2611.04405v1/chunks/ea43f2ef2e2f4cf6b434637410c22ada.json


{
  "status": "chunked",
  "chunk_count": 50,
  "pages_without_text": [],
  "configuration": {
    "method": "page_word_windows_v1",
    "max_words": 100,
    "overlap_words": 20,
    "word_definition": "non-whitespace runs",
    "offsets": "Unicode character indices, end-exclusive",
    "cross_page": false
  },
  "output": "data/arxiv/2611.04405v1/chunks/ea43f2ef2e2f4cf6b434637410c22ada.json"
}
First chunk; its exact source-page slice was checked:
{
  "chunk_id": "9ae7df4122b58fb74a4d284eda91437e338bdf6008cfccb74223b578ebb5c871",
  "arxiv_id": "2611.04405",
  "version": 1,
  "source_pdf_sha256": "932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075",
  "page_number": 1,
  "chunk_number": 1,
  "start_char": 4,
  "end_char": 1363,
  "text": "DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for\n    Imbalanced Diabetic Retinopathy Grading via Adaptive SMOTE Fusion\n\n\n                                Nishi Kanta Paul    Md Shihabul Islam Shovo\n                     

## 5. Inspect one page

Change `PAGE_NUMBER` and rerun only this cell to inspect a different page. These are physical PDF pages, which may differ from printed page labels. A nonempty extraction does not by itself prove correct reading order or scientific relevance.

In [6]:
PAGE_NUMBER = 1
page = extraction_result["pages"][PAGE_NUMBER - 1]
document = extraction_result["download"]["search"]["document"]
print(f"Source: {document['arxiv_id']}v{document['version']}, physical page {page['page_number']}")
print(f"Source URL: {document['source_url']}")
print(f"PDF SHA-256: {extraction_result['download']['sha256']}")
print(f"Page status: {page['status']}; NUL replacements: {page['nul_replacements']}")
print("\nText preview (first 500 characters):\n")
print(page["text"][:500])

Source: 2611.04405v1, physical page 1
Source URL: http://arxiv.org/abs/2611.04405v1
PDF SHA-256: 932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075
Page status: extracted; NUL replacements: 0

Text preview (first 500 characters):

    DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for
    Imbalanced Diabetic Retinopathy Grading via Adaptive SMOTE Fusion


                                Nishi Kanta Paul    Md Shihabul Islam Shovo
                                 NIMISHES Lab               NIMISHES Lab

                                        Dhaka, Bangladesh                Dhaka, Bangladesh

                                    nishikantapaul108@gmail.com        shihabul900@gmail.com

                         


## 6. Prepare local PostgreSQL

The next cell creates `data/postgres/password.txt` only if it does not exist. It prints the path and status, never the password. The file is excluded from Git. No `.env` file is required.

Docker Compose starts a separate local PostgreSQL container and enables pgvector on first initialization. The first start may download the image. The database port is bound to `127.0.0.1` only. An existing password file is preserved; changing it does not update a password already stored in the database volume.

In [7]:
created = create_password_file()
print(f"Password file: {PASSWORD_FILE}")
print("Created; contents hidden." if created else "Already exists; kept unchanged; contents hidden.")
command = ["docker", "compose", "up", "-d", "--wait", "--wait-timeout", "60", "postgres"]
print("Running:", " ".join(command), flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    exit_code = process.wait()
print(f"Docker process exit code: {exit_code}")
if exit_code != 0:
    raise RuntimeError("PostgreSQL startup failed. Check Docker Desktop and the output above.")

Password file: data/postgres/password.txt
Already exists; kept unchanged; contents hidden.
Running: docker compose up -d --wait --wait-timeout 60 postgres


 Container search4cure-local-postgres-1  Running


 Container search4cure-local-postgres-1  Waiting


 Container search4cure-local-postgres-1  Healthy


Docker process exit code: 0


## 7. Check the actual database

This read-only SQL query reports the connected database, PostgreSQL and pgvector versions, and the number of public tables. It does not import documents or create application tables. Connection and statement waits are limited to five seconds each. Credentials are not included in the result.

In [8]:
print("Read-only connection query:")
print(HEALTH_SQL.strip())
print(json.dumps(check_connection(), indent=2))

Read-only connection query:
SELECT current_database() AS database,
       current_user AS db_user,
       current_setting('server_version') AS postgres_version,
       (SELECT extversion FROM pg_extension WHERE extname = 'vector') AS pgvector_version,
       (SELECT count(*) FROM pg_tables WHERE schemaname = 'public') AS public_table_count,
       1 AS check_value;
Connecting to local PostgreSQL at 127.0.0.1:55432


{
  "database": "search4cure",
  "db_user": "search4cure",
  "postgres_version": "16.15 (Debian 16.15-1.pgdg12+2)",
  "pgvector_version": "0.8.7",
  "public_table_count": 1,
  "check_value": 1
}


## 8. Store one document version and display its SQL metadata

Create the document table explicitly, then import the real download record from this run. Each `(arxiv_id, version)` can occur only once. The importer verifies the PDF fingerprint and size. Importing the same version and content again returns `unchanged`; a changed hash raises a conflict and preserves the stored row.

The SQL row keeps the first imported download snapshot. Later retrieval attempts remain in local JSON files. Source submission times are not exact public-availability times; unknown times stay `NULL`. The database storage time describes our own write.

A pandas DataFrame displays the SQL result in Python memory. This does not store page text, chunks or vectors in SQL. The download status does not assert successful parsing.

In [9]:
import pandas as pd

initialize_schema()
print("Document table initialized.")
started = time.perf_counter()
first_import = store_document(download_result["metadata_path"])
first_seconds = time.perf_counter() - started
started = time.perf_counter()
second_import = store_document(download_result["metadata_path"])
second_seconds = time.perf_counter() - started
print(f"Import durations (single local observation): {first_seconds:.6f}s; repeat {second_seconds:.6f}s")
print("First import in this run:", json.dumps(first_import))
print("Repeated import:", json.dumps(second_import))
assert second_import["status"] == "unchanged"
row = read_document(first_import["arxiv_id"], first_import["version"])
assert row["sha256"] == download_result["sha256"]
print("Parameterized SQL (values are passed separately):")
print(DOCUMENT_SQL.strip())
documents_df = pd.DataFrame([row])
print("Rows returned for this document version:", len(documents_df))
assert len(documents_df) == 1
print(documents_df[["arxiv_id", "version", "status", "size_bytes"]].to_string(index=False))
display(documents_df[["arxiv_id", "version", "title", "status", "pdf_retrieved_at", "stored_at"]])

Connecting to local PostgreSQL at 127.0.0.1:55432


Document table initialized.
Connecting to local PostgreSQL at 127.0.0.1:55432


Document metadata unchanged: 2611.04405v1


Connecting to local PostgreSQL at 127.0.0.1:55432


Document metadata unchanged: 2611.04405v1


Import durations (single local observation): 0.014409s; repeat 0.011202s
First import in this run: {"status": "unchanged", "arxiv_id": "2611.04405", "version": 1, "sha256": "932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075"}
Repeated import: {"status": "unchanged", "arxiv_id": "2611.04405", "version": 1, "sha256": "932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075"}
Connecting to local PostgreSQL at 127.0.0.1:55432


Parameterized SQL (values are passed separately):
SELECT arxiv_id, version, title, source_url, pdf_url, first_submitted_at,
       version_submitted_at, metadata_retrieved_at, pdf_retrieved_at,
       sha256, size_bytes, local_path, metadata_path, status, stored_at
FROM document_versions
WHERE arxiv_id = %s AND version = %s;
Rows returned for this document version: 1
  arxiv_id  version     status  size_bytes
2611.04405        1 downloaded      577377


,arxiv_id,version,title,status,pdf_retrieved_at,stored_at
0,2611.04405,1,DR-LiteNet: A Lightweight Explainable Hybrid C...,downloaded,2026-11-04 16:36:49.980198+00:00,2026-11-04 16:36:51.570138+00:00


## 9. Run the tests

API and PDF tests use synthetic responses and generated files. Opt-in integration tests contact the actual PostgreSQL database. Document tests use explicitly synthetic download records in isolated schemas that are rolled back after each test; they do not add synthetic documents to the application table. Expected synthetic network-failure warnings exercise error handling. The final unittest result determines success.

In [10]:
command = [
    sys.executable, "-u", "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_*.py", "-v",
]
print("Running offline tests and actual PostgreSQL integration tests...", flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env={**os.environ, "S4C_TEST_POSTGRES": "1"}) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    exit_code = process.wait()
print(f"Test process exit code: {exit_code}")
if exit_code != 0:
    raise RuntimeError("The tests failed.")

Running offline tests and actual PostgreSQL integration tests...


test_empty_and_invalid_entries (test_arxiv_search.ArxivSearchTests.test_empty_and_invalid_entries) ... ok


test_rate_limit_and_retry_after_stop_automatic_retries (test_arxiv_search.ArxivSearchTests.test_rate_limit_and_retry_after_stop_automatic_retries) ... WARNING:root:arXiv request failed: attempt=1/3 error=HTTP Error 429: Synthetic HTTP error


ok


test_real_request_shape_limits_results_and_sets_timeout (test_arxiv_search.ArxivSearchTests.test_real_request_shape_limits_results_and_sets_timeout) ... ok


test_retries_stop_after_three_attempts (test_arxiv_search.ArxivSearchTests.test_retries_stop_after_three_attempts) ... WARNING:root:arXiv request failed: attempt=1/3 error=<urlopen error Synthetic connection failure>


... (51 outputs left out)

----------------------------------------------------------------------


Ran 50 tests in 0.701s


OK


Test process exit code: 0


## Current limits

OCR, image/table interpretation, embeddings, SQL storage of page text/chunks, document search, answer generation, and AWS deployment are not implemented. Parser sorting approximates reading order; extraction quality requires a separate content check. A parser-repaired PDF is flagged. Public availability cannot be inferred solely from arXiv submission timestamps.

SQL stores the first successful download snapshot per document version. Repeated download attempts remain in JSON files. The `downloaded` status is not a parsing or embedding status. Import-run tracking and SQL failure/processing records are later steps. Backup and restore have not been tested.

Each future development step extends this same notebook after its function and tests are implemented.